# Batch 7b: inverse PINN constants for Ghezzi's closed-loop models (2-DOF)
Same tasks, flights and equation as Batch 7: y'' = wn^2 (r(t - tau) - y) - 2 xi wn y'.
The inverse PINN (as in Batch 2): a network NN(t) -> y for one train flight, plus trainable wn, xi; loss = physics + w_data * data. One PINN per train flight, constants averaged. Delay tau fixed to the Batch 7 refit value. The constants are then scored on the test flights with the same simulation as the refit.

In [1]:
import numpy as np, pandas as pd, torch, time
from scipy.integrate import solve_ivp
torch.set_num_threads(4)
D = '../data/clean/2dof/'
grid = lambda n: pd.read_csv(D + n + '.csv').iloc[::5].reset_index(drop=True)
def att_window(f):
    k = np.flatnonzero(np.abs(f.theta_sp.values) > 0.05)[0]; t0 = f.t[k]
    return f[(f.t >= t0 - 1) & (f.t < t0 + 3)].reset_index(drop=True)
K = pd.read_csv('../results/batch7_ghezzi_constants.csv')
ref = lambda task, col: float(K[(K.task == task) & (K.model.str.startswith('grey'))][col].iloc[0].split(' ')[0])
TASKS = {
 'CL attitude': dict(y='theta', yd='q', r='theta_sp', train=['attitude_doublet_1', 'attitude_doublet_3'], test=['attitude_doublet_4', 'attitude_doublet_5'],
                     cut=att_window, start=(16.16, 0.56), scale=0.3, width=32, epochs=30000),
 'CL position': dict(y='x', yd='v', r='x_sp', train=['position_1', 'position_3'], test=['position_2'],
                     cut=lambda f: f, start=(3.47, 0.84), scale=0.3, width=64, epochs=30000)}
for task, c in TASKS.items(): c['tau'] = ref(task, 'tau'); c['refit'] = (ref(task, 'wn'), ref(task, 'xi'))
F = {n: c['cut'](grid(n)) for c in TASKS.values() for n in c['train'] + c['test']}
vaf = lambda y, yh: max(1 - np.var(y - yh) / np.var(y), 0) * 100
def accel(y, yd, r, p): return p['wn']**2 * (r - y) - 2 * p['xi'] * p['wn'] * yd
def run_ivp(p, f, c):
    t, R = f.t.values, f[c['r']].values
    rhs = lambda tt, z: [z[1], accel(z[0], z[1], np.interp(tt - c['tau'], t, R), p)]
    return solve_ivp(rhs, (t[0], t[-1]), [f[c['y']][0], f[c['yd']][0]], t_eval=t, max_step=0.005, rtol=1e-8).y

In [2]:
def mlp(n_in, n_out, width=32, depth=3):
    layers, n = [], n_in
    for _ in range(depth):
        layers += [torch.nn.Linear(n, width), torch.nn.Tanh()]; n = width
    return torch.nn.Sequential(*layers, torch.nn.Linear(n, n_out))
col = lambda x: torch.tensor(np.asarray(x), dtype=torch.float32).reshape(-1, 1)

def train_inverse_pinn(f, c, seed=0, lr=1e-3, w_data=3000.0):
    # data
    torch.manual_seed(seed)
    t_np = f.t.values - f.t.values[0]; T = t_np[-1]
    t_all = col(t_np); r_del = col(np.interp(t_np - c['tau'], t_np, f[c['r']].values)); y_all = col(f[c['y']].values)
    # network: t -> y
    net = mlp(1, 1, c['width']); y_of = lambda t: c['scale'] * net(2 * t / T - 1)
    raw = {k: torch.nn.Parameter(torch.tensor(float(np.log(v)))) for k, v in zip(['wn', 'xi'], c['start'])}
    consts = lambda: {k: v.exp() for k, v in raw.items()}
    # optimizer
    opt = torch.optim.Adam(list(net.parameters()) + list(raw.values()), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, c['epochs'], eta_min=lr / 20)
    # loop
    for e in range(c['epochs']):
        t = t_all.clone().requires_grad_(True); y = y_of(t)
        yd = torch.autograd.grad(y, t, torch.ones_like(y), create_graph=True)[0]
        ydd = torch.autograd.grad(yd, t, torch.ones_like(yd), create_graph=True)[0]
        physics = (((ydd - accel(y, yd, r_del, consts())) / c['scale'])**2).mean()
        data = (((y - y_all) / c['scale'])**2).mean()
        loss = physics + w_data * data
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    return dict(p={k: float(v.detach()) for k, v in consts().items()}, physics=physics.item(), data=data.item())

In [3]:
rows, consts = [], []
for task, c in TASKS.items():
    P = []
    for n in c['train']:
        t0 = time.time(); r = train_inverse_pinn(F[n], c); P.append(r['p'])
        print(task, n, {k: round(v, 3) for k, v in r['p'].items()}, f"data {r['data']:.2e} physics {r['physics']:.2e} ({time.time() - t0:.0f} s)")
    p = {k: np.mean([q[k] for q in P]) for k in ['wn', 'xi']}; sd = {k: np.std([q[k] for q in P]) for k in ['wn', 'xi']}
    consts.append(dict(task=task, model='inverse PINN (mean of train flights)', wn=f"{p['wn']:.3f} +- {sd['wn']:.3f}", xi=f"{p['xi']:.3f} +- {sd['xi']:.3f}", tau=c['tau']))
    for n in c['test']:
        f = F[n]
        for label, pp in [('inverse PINN constants', p), ('grey-box refit constants', dict(wn=c['refit'][0], xi=c['refit'][1]))]:
            Y = run_ivp(pp, f, c)
            rows.append(dict(task=task, test=n, model=label, VAF_y=vaf(f[c['y']].values, Y[0]), VAF_rate=vaf(f[c['yd']].values, Y[1])))
T = pd.DataFrame(rows).round(2); T.to_csv('../results/batch7b_inverse_pinn_vaf.csv', index=False)
KK = pd.DataFrame(consts); KK.to_csv('../results/batch7b_inverse_pinn_constants.csv', index=False)
display(KK); T

CL attitude attitude_doublet_1 {'wn': 23.608, 'xi': 0.884} data 1.51e-03 physics 1.87e-01 (67 s)


CL attitude attitude_doublet_3 {'wn': 29.2, 'xi': 0.903} data 5.18e-04 physics 4.27e-01 (69 s)


CL position position_1 {'wn': 1.034, 'xi': 0.407} data 7.14e-05 physics 1.55e-01 (145 s)


CL position position_3 {'wn': 1.501, 'xi': 0.694} data 1.87e-05 physics 8.81e-02 (129 s)


,task,model,wn,xi,tau
0,CL attitude,inverse PINN (mean of train flights),26.404 +- 2.796,0.894 +- 0.009,0.022
1,CL position,inverse PINN (mean of train flights),1.268 +- 0.234,0.550 +- 0.143,0.003


,task,test,model,VAF_y,VAF_rate
0,CL attitude,attitude_doublet_4,inverse PINN constants,99.46,87.78
1,CL attitude,attitude_doublet_4,grey-box refit constants,99.46,86.36
2,CL attitude,attitude_doublet_5,inverse PINN constants,99.59,91.05
3,CL attitude,attitude_doublet_5,grey-box refit constants,99.60,89.77
4,CL position,position_2,inverse PINN constants,97.00,79.95
5,CL position,position_2,grey-box refit constants,99.28,91.78
